In [43]:
import json
from pathlib import Path
import pandas as pd

greenhouse = json.loads(Path("../samples/greenhouse/axonius.json").read_text(encoding="utf-8"))
gh = pd.json_normalize(greenhouse["jobs"])
gh[["id", "title", "location.name"]]





,id,title,location.name
0,7792094003,"Area Sales Director, Southwest",Remote US
1,8181415003,Director of Product Security,"Tel Aviv, Israel"
2,7995556003,"Director, Product Design","Tel Aviv, Israel"
3,7820903003,"Enterprise Account Executive, Carolinas",Remote US
4,7821059003,"Enterprise Account Executive, Florida",Remote US
5,7815968003,"Enterprise Account Executive, Ohio/Michigan",Remote US
6,8001611003,"Manager, Financial Systems",Remote US
7,8001615003,Principal IT Architect,Remote US
8,8004240003,Product Security Architect- 9-month temp position,Portugal
9,7886132003,Product Security Architect- 9-month temp position,"Tel Aviv, Israel"


In [44]:
gh.columns

Index(['absolute_url', 'data_compliance', 'internal_job_id', 'metadata', 'id',
       'updated_at', 'requisition_id', 'title', 'company_name',
       'first_published', 'language', 'application_deadline', 'content',
       'departments', 'offices', 'ai_disclaimer', 'include_ai_disclaimer',
       'ai_opt_out_request_url', 'location.name'],
      dtype='str')

In [45]:
import json
from pathlib import Path
import pandas as pd

comeet = json.loads(Path("../samples/comeet/kmslh.json").read_text(encoding="utf-8"))
cm = pd.json_normalize(comeet)
cm.columns





Index(['name', 'department', 'email', 'url_comeet_hosted_page',
       'url_recruit_hosted_page', 'url_active_page', 'employment_type',
       'experience_level', 'uid', 'internal_use_custom_id',
       'url_detected_page', 'picture_url', 'time_updated', 'company_name',
       'is_internal', 'linkedin_job_posting_id', 'workplace_type',
       'location.name', 'location.country', 'location.city', 'location.state',
       'location.postal_code', 'location.street_name',
       'location.arrival_instructions', 'location.street_number',
       'location.timezone', 'location.location_uid', 'location.is_remote',
       'custom_fields.details', 'custom_fields.categories'],
      dtype='str')

In [46]:
cm[["uid","name","workplace_type"]]

,uid,name,workplace_type
0,9D.073,DevOps Engineer,Hybrid
1,FA.D61,Enterprise Account Executive,Hybrid
2,95.478,Global Learning & Training Team Lead,On-site
3,CA.47D,Product Manager,On-site
4,95.F68,R&D Director,On-site
5,AC.47A,Support Engineer,Hybrid
6,4F.27D,Support Engineer Team Lead,On-site


In [47]:
gh_il = gh[gh["location.name"].str.contains("Israel")]
gh_il[["id", "title", "location.name"]]

,id,title,location.name
1,8181415003,Director of Product Security,"Tel Aviv, Israel"
2,7995556003,"Director, Product Design","Tel Aviv, Israel"
9,7886132003,Product Security Architect- 9-month temp position,"Tel Aviv, Israel"
12,8044523003,Senior AI Product Manager,"Tel Aviv, Israel"
13,7995553003,Senior Cyber Data Analyst,"Tel Aviv, Israel"
16,8008023003,Senior Product Manager,"Tel Aviv, Israel"
18,7809799003,Senior Security Researcher,"Tel Aviv, Israel"


In [48]:
cm_il = cm[cm["location.country"] == "IL"]
cm_il[["uid", "name", "location.city", "workplace_type"]]

,uid,name,location.city,workplace_type
2,95.478,Global Learning & Training Team Lead,Petah Tikva,On-site
3,CA.47D,Product Manager,Petah Tikva,On-site
4,95.F68,R&D Director,Petah Tikva,On-site


In [49]:
import hashlib
import html
import re

YEARS_RE = re.compile(r"(\d+)(?:\s*[-–]\s*\d+)?\+?\s*years?", re.IGNORECASE)


def html_to_text(raw):
    text = re.sub(r"<[^>]+>", " ", raw)
    return re.sub(r"\s+", " ", html.unescape(text))


def extract_min_years(text):
    match = YEARS_RE.search(text)
    return int(match.group(1)) if match else None


def comeet_description(details):
    return "".join(f"<h3>{s['name']}</h3>{s['value']}" for s in details or [] if s["value"])


def make_content_hash(row):
    parts = [row["title"], row["city"], row["description"]]
    text = "|".join(str(p) for p in parts)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [50]:
gh_description = gh_il["content"].apply(html.unescape)

gh_unified = pd.DataFrame({
    "ats_type": "greenhouse",
    "company_id": "515673382",
    "external_id": gh_il["id"].astype(str),
    "title": gh_il["title"],
    "city": gh_il["location.name"].str.split(",").str[0].str.strip(),
    "level": None,
    "work_mode": None,
    "min_years_experience": gh_description.apply(html_to_text).apply(extract_min_years).astype("Int64"),
    "url": gh_il["absolute_url"],
    "description": gh_description,
})
gh_unified[["title", "city", "min_years_experience"]]

,title,city,min_years_experience
1,Director of Product Security,Tel Aviv,12
2,"Director, Product Design",Tel Aviv,4
9,Product Security Architect- 9-month temp position,Tel Aviv,3
12,Senior AI Product Manager,Tel Aviv,5
13,Senior Cyber Data Analyst,Tel Aviv,4
16,Senior Product Manager,Tel Aviv,5
18,Senior Security Researcher,Tel Aviv,4


In [51]:
cm_description = cm_il["custom_fields.details"].apply(comeet_description)

cm_unified = pd.DataFrame({
    "ats_type": "comeet",
    "company_id": "512336231",
    "external_id": cm_il["uid"],
    "title": cm_il["name"],
    "city": cm_il["location.city"],
    "level": cm_il["experience_level"],
    "work_mode": cm_il["workplace_type"],
    "min_years_experience": cm_description.apply(html_to_text).apply(extract_min_years).astype("Int64"),
    "url": cm_il["url_active_page"],
    "description": cm_description,
})
cm_unified[["title", "city", "level", "work_mode", "min_years_experience"]]

,title,city,level,work_mode,min_years_experience
2,Global Learning & Training Team Lead,Petah Tikva,Intermediate,On-site,<NA>
3,Product Manager,Petah Tikva,NaN,On-site,3
4,R&D Director,Petah Tikva,Senior,On-site,10


In [52]:
jobs = pd.concat([gh_unified, cm_unified], ignore_index=True)
jobs["job_id"] = jobs["ats_type"] + ":" + jobs["company_id"] + ":" + jobs["external_id"]
jobs["content_hash"] = jobs.apply(make_content_hash, axis=1)
jobs[["job_id", "title", "level", "work_mode", "min_years_experience", "content_hash"]]

,job_id,title,level,work_mode,min_years_experience,content_hash
0,greenhouse:515673382:8181415003,Director of Product Security,None,None,12,1330acce9bbb7e42ba4a7ed34f1c702d3298dff634d30a...
1,greenhouse:515673382:7995556003,"Director, Product Design",None,None,4,207201b83c016010bb12d8e5e425824511ebbf1c8c6cd7...
2,greenhouse:515673382:7886132003,Product Security Architect- 9-month temp position,None,None,3,cf79783964ce59ad11bfcdec1768e038919bdb03e0373c...
3,greenhouse:515673382:8044523003,Senior AI Product Manager,None,None,5,e2407c7179228dba21d01a5616ea1b6cbab785f3d6b90b...
4,greenhouse:515673382:7995553003,Senior Cyber Data Analyst,None,None,4,e43fcba4a60fdd96b78b232cf07d03df42f2c17ce2bb85...
5,greenhouse:515673382:8008023003,Senior Product Manager,None,None,5,a4437a1d5f5dd91a3afb90c310153c4c3c9933f41c8a98...
6,greenhouse:515673382:7809799003,Senior Security Researcher,None,None,4,a5533f8fa16cb6c511c00194d90638d1e23cb8e3e215cc...
7,comeet:512336231:95.478,Global Learning & Training Team Lead,Intermediate,On-site,<NA>,9625b26f18d81ecb6d8c7b64b7a43c22fb74c09969cfbf...
8,comeet:512336231:CA.47D,Product Manager,NaN,On-site,3,9121224184fccecc775c4858ec22f07eead8785abcafd9...
9,comeet:512336231:95.F68,R&D Director,Senior,On-site,10,6e49964ed57cfbc40fccbac9c42a6d2426059472ff26cb...


In [53]:
import os

import psycopg
from dotenv import load_dotenv

load_dotenv("../.env")

records = jobs.astype(object).where(jobs.notna(), None).to_dict("records")

with psycopg.connect(
    host="localhost",
    port=5432,
    user="noam",
    password=os.environ["POSTGRES_PASSWORD"],
    dbname="jobradar",
) as conn:
    for job in records:
        conn.execute(
            """
            INSERT INTO job_postings (job_id, company_id, ats_type, external_id, title, level,
                                      city, work_mode, url, description, min_years_experience,
                                      content_hash)
            VALUES (%(job_id)s, %(company_id)s, %(ats_type)s, %(external_id)s, %(title)s, %(level)s,
                    %(city)s, %(work_mode)s, %(url)s, %(description)s, %(min_years_experience)s,
                    %(content_hash)s)
            ON CONFLICT (job_id) DO NOTHING
            """,
            job,
        )

print("Processed:", len(records))


Processed: 10
